### Question 2

# Task 6: Define the Modelling Strategy

*   **Name:** Jamaicah JACOB
*   **Student ID:** 202210047
*   **Part A Group:** [Insert Your Part A Group Number/Name]
*   **Assigned Member Number:** 3
*   **Assigned Dataset Version:** ZP

### Question 3

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing & Pipelines
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Model Selection & Evaluation
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, make_scorer

# Assigned Classifiers (REPLACE THESE WITH YOUR ASSIGNED 4 MODELS)
# Example: 
# from sklearn.linear_model import LogisticRegression
# from sklearn.ensemble import RandomForestClassifier
# from xgboost import XGBClassifier
# from sklearn.svm import SVC

### Question 4

In [2]:
# Load data
X_train = pd.read_csv(r"C:\Users\CLIENT\Downloads\X_train.csv")
y_train = pd.read_csv(r"C:\Users\CLIENT\Downloads\y_train.csv")

# 1. Confirm matching row counts and alignment
print(f"X_train rows: {X_train.shape[0]}, y_train rows: {y_train.shape[0]}")
assert X_train.shape[0] == y_train.shape[0], "Mismatch in row counts!"

# 2. Confirm 20 predictor columns
print(f"Number of predictors: {X_train.shape[1]}")
assert X_train.shape[1] == 20, f"Expected 20 predictors, found {X_train.shape[1]}!"

# 3. Confirm binary target values
target_unique = set(y_train.iloc[:, 0].dropna().unique())
print(f"Target unique values: {target_unique}")
assert target_unique == {0, 1}, f"Target is not strictly binary {0, 1}: found {target_unique}"

# 4. Identify feature types using supplied feature schema
numerical_features = [
    'longitude', 'latitude', 'number_of_vehicles', 
    'number_of_casualties', 'casualties_per_vehicle'
]

ordinal_features = [
    'speed_limit', 'first_road_class_ordinal', 'second_road_class_ordinal'
]

nominal_features = [
    'day_of_week', 'road_type', 'junction_detail', 'junction_control', 
    'light_conditions', 'weather_conditions', 'road_surface_conditions', 
    'month', 'time_period'
]

binary_features = [
    'urban_or_rural_area', 'has_second_road', 'second_road_unknown'
]

# Verify feature classification covers all 20 columns exactly
all_categorized_features = numerical_features + ordinal_features + nominal_features + binary_features
assert len(all_categorized_features) == 20, f"Categorized {len(all_categorized_features)} features, expected 20!"

# 5. Report missing values by feature
print("\nMissing values per feature:")
print(X_train.isnull().sum())

X_train rows: 8000, y_train rows: 8000
Number of predictors: 20
Target unique values: {np.int64(0), np.int64(1)}

Missing values per feature:
longitude                       0
latitude                        0
number_of_vehicles              0
number_of_casualties            0
day_of_week                     0
road_type                       0
speed_limit                     0
junction_detail               563
junction_control             3426
light_conditions                0
weather_conditions              0
road_surface_conditions        57
urban_or_rural_area             0
month                           0
time_period                     0
casualties_per_vehicle          0
has_second_road                 0
second_road_unknown             0
first_road_class_ordinal        0
second_road_class_ordinal       0
dtype: int64


### Question 5

In [3]:
# 1. Report target counts and proportions
target_counts = y_train.iloc[:, 0].value_counts()
target_proportions = y_train.iloc[:, 0].value_counts(normalize=True)

print("--- Training Target Counts ---")
print(target_counts)

print("\n--- Training Target Proportions ---")
print(target_proportions)

# 2. Identify majority class and calculate majority-class accuracy
majority_class = target_counts.idxmax()
majority_class_count = target_counts.max()
total_samples = len(y_train)

majority_class_accuracy = target_proportions.max()

print(f"\nMajority Class: {majority_class}")
print(f"Majority Class Count: {majority_class_count} out of {total_samples}")
print(f"Majority-Class Accuracy: {majority_class_accuracy:.4f} ({majority_class_accuracy * 100:.2f}%)")

--- Training Target Counts ---
is_severe_collision
0    6013
1    1987
Name: count, dtype: int64

--- Training Target Proportions ---
is_severe_collision
0    0.751625
1    0.248375
Name: proportion, dtype: float64

Majority Class: 0
Majority Class Count: 6013 out of 8000
Majority-Class Accuracy: 0.7516 (75.16%)


### Question 5: Training Target Summary & Baseline Analysis

#### 1. Target Class Counts & Proportions
- **Class 0 (Non-Severe Collision):** 6,013 samples (**75.16%**)
- **Class 1 (Severe Collision):** 1,987 samples (**24.84%**)
- **Total Training Samples:** 8,000

#### 2. Majority Class & Baseline Accuracy
- **Majority Class:** `0`
- **Majority-Class Accuracy:** **75.16%** (`0.7516`)

---

#### Written Interpretation & Insights
* **Baseline Accuracy:** A naive model that predicts the majority class (`0`) for every sample will achieve **75.16% accuracy** simply by chance.
* **Implication for Model Evaluation:** Despite a high accuracy score, a majority-class classifier completely fails to detect severe collisions (**Class 1 Recall = 0.0**, **Class 1 Precision = 0.0**, **Class 1 F1-score = 0.0**). This demonstrates why **Accuracy alone is a misleading metric** for this task and reinforces the choice of **F1-score for Class 1** as our primary selection metric.

### Question 6 & 7

In [4]:
# ==========================================
# Question 7: Define 5-Fold Stratified Cross-Validation
# ==========================================
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Define evaluation metrics
scoring_metrics = {
    'f1': 'f1',                  # Primary metric (Class 1 F1-Score)
    'accuracy': 'accuracy',      # Secondary metric
    'precision': 'precision',    # Secondary metric
    'recall': 'recall',          # Secondary metric
    'roc_auc': 'roc_auc'         # Secondary metric
}

# ==========================================
# Question 6: DummyClassifier Minimum Baseline
# ==========================================
dummy_clf = DummyClassifier(strategy='most_frequent')

# Evaluate DummyClassifier using cross_validate
dummy_results = cross_validate(
    dummy_clf, 
    X_train, 
    y_train.iloc[:, 0], 
    cv=cv, 
    scoring=scoring_metrics
)

# Summarize baseline performance across 5 folds
baseline_metrics_df = pd.DataFrame({
    'Metric': ['F1-Score (Class 1)', 'Accuracy', 'Precision', 'Recall', 'ROC-AUC'],
    'Mean CV Score': [
        dummy_results['test_f1'].mean(),
        dummy_results['test_accuracy'].mean(),
        dummy_results['test_precision'].mean(),
        dummy_results['test_recall'].mean(),
        dummy_results['test_roc_auc'].mean()
    ],
    'Std CV Score': [
        dummy_results['test_f1'].std(),
        dummy_results['test_accuracy'].std(),
        dummy_results['test_precision'].std(),
        dummy_results['test_recall'].std(),
        dummy_results['test_roc_auc'].std()
    ]
})

print("--- DummyClassifier Baseline Cross-Validation Results ---")
print(baseline_metrics_df.to_string(index=False))

c:\Users\CLIENT\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\CLIENT\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\CLIENT\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


--- DummyClassifier Baseline Cross-Validation Results ---
            Metric  Mean CV Score  Std CV Score
F1-Score (Class 1)       0.000000      0.000000
          Accuracy       0.751625      0.000306
         Precision       0.000000      0.000000
            Recall       0.000000      0.000000
           ROC-AUC       0.500000      0.000000


c:\Users\CLIENT\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\CLIENT\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


### Baseline Model Insights

- **Baseline Metrics Summary:**
  - **Class 1 F1-Score:** `0.0000` (Minimum benchmark)
  - **Accuracy:** `0.7516` (75.16%)
  - **Precision & Recall:** `0.0000`
  - **ROC-AUC:** `0.5000` (Equivalent to random guessing)

- **Key Takeaway:** 
  The `DummyClassifier` achieves **75.16% accuracy** simply by predicting the majority class (`0`) for every collision. However, its **F1-score, Precision, and Recall for Class 1 are all 0.0000** because it fails to identify a single severe collision. Any candidate model developed in subsequent steps must achieve a **Class 1 F1-Score greater than 0.0000** to demonstrate genuine predictive utility over this baseline.

### Question 8 & 9

### Task 6.8: Primary Selection Metric — F1-Score for Class 1

**Primary Metric:** `F1-Score` (specifically for positive target Class 1: Severe Collisions).

#### Why Identifying Severe Collisions Requires Precision AND Recall
Evaluating severe collision predictions involves balancing two distinct costs of misclassification:

1. **False Negatives (Low Recall):** Occurs when a severe collision is misclassified as non-severe. 
   - *Consequence:* High real-world risk. Safety authorities fail to allocate timely emergency medical care, traffic management, or infrastructure intervention to high-risk crash locations/scenarios.
2. **False Positives (Low Precision):** Occurs when a non-severe collision is misclassified as severe.
   - *Consequence:* High resource/economic cost. Unnecessary emergency responses, false alarms, and wasted computational/budgetary resources.

#### The Role of the F1-Score
- **Precision** measures exactness: $\text{Precision} = \frac{\text{TP}}{\text{TP} + \text{FP}}$
- **Recall** measures completeness: $\text{Recall} = \frac{\text{TP}}{\text{TP} + \text{FN}}$
- **F1-Score** is the harmonic mean of Precision and Recall:
  $$\text{F1-Score} = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$

Using the harmonic mean penalizes extreme imbalances between Precision and Recall. Selecting models based on F1-Score ensures that the chosen model achieves a practical, effective balance—maximizing severe collision detection while suppressing costly false alarms.

---

### Task 6.9: Secondary Metrics & The Limitation of Accuracy

To provide a holistic view of model performance, four secondary metrics are recorded across all cross-validation evaluations:

1. **Accuracy:** Overall proportion of correct predictions ($\frac{\text{TP} + \text{TN}}{\text{Total}}$).
2. **Precision:** Accuracy of positive (severe) predictions.
3. **Recall:** Proportion of actual severe collisions correctly identified.
4. **ROC-AUC (Receiver Operating Characteristic - Area Under Curve):** Measures model discrimination across all decision thresholds, independent of a fixed 0.5 probability cutoff.

#### Why Models MUST NOT Be Selected Using Accuracy Alone
- **Sensitivity to Class Imbalance:** As established in Task 6.5, our dataset exhibits a **75.16% / 24.84%** class distribution.
- **The "Accuracy Paradox":** A completely uninformative baseline model (`DummyClassifier`) that blindly predicts `0` for every instance achieves **75.16% Accuracy**. 
- If model selection were based on accuracy alone, a useless model with **0% Recall** for severe collisions could rank higher than a useful predictive model that achieves 73% accuracy with an 80% recall for severe accidents. Accuracy reflects performance on the dominant majority class while masking complete failure on the minority class of interest.

In [5]:
from sklearn.metrics import make_scorer, f1_score, precision_score, recall_score, accuracy_score, roc_auc_score

# 1. Define custom scoring functions with zero_division=0 handling
f1_primary = make_scorer(f1_score, pos_label=1, zero_division=0)
precision_sec = make_scorer(precision_score, pos_label=1, zero_division=0)
recall_sec = make_scorer(recall_score, pos_label=1, zero_division=0)

# 2. Construct evaluation dictionary combining primary and secondary metrics
evaluation_metrics = {
    'primary_f1': f1_primary,
    'accuracy': 'accuracy',
    'precision': precision_sec,
    'recall': recall_sec,
    'roc_auc': 'roc_auc'
}

# 3. Helper function to compile and present cross-validation results neatly
def evaluate_cv_performance(model_name, cv_results):
    """
    Formats 5-fold cross-validation output dictionary into a clean summary DataFrame.
    """
    summary = pd.DataFrame({
        'Model': model_name,
        'Primary F1-Score': [cv_results['test_primary_f1'].mean()],
        'Accuracy': [cv_results['test_accuracy'].mean()],
        'Precision': [cv_results['test_precision'].mean()],
        'Recall': [cv_results['test_recall'].mean()],
        'ROC-AUC': [cv_results['test_roc_auc'].mean()]
    })
    return summary

print("Evaluation metric dictionary and reporting helper function successfully configured.")

Evaluation metric dictionary and reporting helper function successfully configured.
